# Notebook 5 — Agentic RAG: Planning Before Retrieving

All the previous notebooks followed the same basic flow:

```
Question → Retrieve → Answer
```

That works for simple questions. But what about this:

*"How does user authentication connect to user creation, and what shared logic do they use?"*

A single retrieval step might find something about auth, or something about user creation — but not necessarily both, and not the connection between them.

To answer this well, you need to:
1. Understand what the question is really asking
2. Decide what pieces of information you need to find
3. Retrieve those pieces separately
4. Check that you actually found useful information
5. Combine everything into a final answer

This is what an agent does. It **plans before it acts**.

**This notebook adds an orchestration layer** on top of retrieval. A planner (Gemini) breaks the question into sub-queries. Each sub-query is run against the indexed repository. The results become named observations. Those observations are verified and then synthesized into a final answer.

**How to run:** set the repo URL in the configuration cell, run all cells, use the Gradio interface.

**API key needed:** `GOOGLE_API_KEY` (Gemini) in Colab Secrets.


In [ ]:
!pip install -q \
    google-generativeai \
    pydantic \
    sentence-transformers \
    qdrant-client \
    gradio


In [ ]:
import json
import os
import time
import re
import subprocess
import shutil
from pathlib import Path
from typing import List, Tuple, Optional

from pydantic import BaseModel, Field
import google.generativeai as genai
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

print("All imports successful.")

All imports successful.


## Configuration

Set the GitHub repository URL here. This is the only thing you need to change.

The notebook will clone the repository, chunk all code files, embed them, and store them in an in-memory Qdrant vector database.

Supported URL formats:
- `https://github.com/username/repo`
- `https://github.com/username/repo.git`

To use a local folder instead, set `REPO_URL = ""` and set `LOCAL_REPO_PATH` to your folder path.


In [ ]:
     
#  SET YOUR REPO URL HERE

REPO_URL = "https://github.com/tiangolo/fastapi"  # ← change this

# If using a local folder instead of GitHub, set REPO_URL = "" and set this:
LOCAL_REPO_PATH = ""  # e.g. "/content/my_project"

# Chunking settings 
CHUNK_SIZE = 400  # characters per chunk
CHUNK_OVERLAP = 80  # overlap between consecutive chunks

# Retrieval settings 
TOP_K = 4  # chunks to retrieve per agent step

# File types to index 
SUPPORTED_EXTENSIONS = {
    ".py",
    ".js",
    ".ts",
    ".jsx",
    ".tsx",
    ".java",
    ".go",
    ".rs",
    ".cpp",
    ".c",
    ".h",
    ".md",
    ".txt",
    ".yaml",
    ".yml",
    ".toml",
    ".json",
    ".html",
    ".css",
    ".sh",
    ".env.example",
}

# Folders to skip 
SKIP_DIRS = {
    ".git",
    "__pycache__",
    "node_modules",
    ".venv",
    "venv",
    "env",
    "dist",
    "build",
    ".idea",
    ".vscode",
    "coverage",
    ".pytest_cache",
    "*.egg-info",
}

print("Configuration loaded.")
print(f"  Repo URL  : {REPO_URL or LOCAL_REPO_PATH}")
print(f"  Chunk size: {CHUNK_SIZE} chars  |  Overlap: {CHUNK_OVERLAP} chars")
print(f"  Top-K     : {TOP_K}")

Configuration loaded.
  Repo URL  : https://github.com/tiangolo/fastapi
  Chunk size: 400 chars  |  Overlap: 80 chars
  Top-K     : 4


## Gemini Setup

This cell connects to the Gemini API.

Gemini is used for two things in this notebook:
1. **Planning** — breaking a complex question into retrieval steps
2. **Synthesis** — combining all observations into a final answer

Your API key is read from Colab Secrets (`GOOGLE_API_KEY`). Get a free key at aistudio.google.com.


In [ ]:
def get_api_key() -> str:
    try:
        from google.colab import userdata

        key = userdata.get("GEMINI_API_KEY")
        if key:
            return key
    except Exception:
        pass
    key = os.environ.get("GEMINI_API_KEY", "")
    if not key:
        raise ValueError(
            "GEMINI_API_KEY not found.\n"
            "In Colab: Secrets panel (🔑) → Add GEMINI_API_KEY\n"
            "Locally : export GEMINI_API_KEY=your_key"
        )
    return key


API_KEY = get_api_key()
genai.configure(api_key=API_KEY)
gemini_model = genai.GenerativeModel("gemini-2.5-flash")
print("Gemini 2.5 Flash ready.")

Gemini 2.5 Flash ready.


## Planning Schemas with Pydantic

The planner outputs a structured plan — a list of retrieval steps with specific queries.

This plan is validated using Pydantic before the agent executes it.

**Why Pydantic?**

The planner returns JSON. Raw JSON has no guarantees — wrong field names, missing fields, or wrong types can cause the agent to fail silently.

Pydantic parses and validates the output automatically. If a step is missing its query field, you get a clear error immediately at planning time, not a confusing failure later.

The plan schema looks like:

```python
class PlanStep:
    step_number: int
    query: str        # what to search for
    rationale: str    # why this step is needed

class AgentPlan:
    objective: str
    steps: List[PlanStep]
    confidence: float
```


In [ ]:
class PlanStep(BaseModel):
    step: int
    action: str
    query: str


class SearchPlan(BaseModel):
    objective: str
    steps: List[PlanStep]
    target_components: List[str]


class Observation(BaseModel):
    step: int
    query: str
    result_summary: str
    chunks_found: int = 0


class PlanExecutionResponse(BaseModel):
    plan: SearchPlan
    confidence_score: float = Field(ge=0.0, le=1.0)


print("Pydantic schemas defined.")

Pydantic schemas defined.


## Embedding Model and Vector Store

This cell loads the embedding model and sets up the in-memory Qdrant vector database.

The embedding model (`all-MiniLM-L6-v2`) converts text into 384-dimensional vectors. Qdrant stores those vectors and does similarity search when the agent retrieves chunks.

The vector store is in-memory, so it resets when the runtime restarts. For persistent storage, replace `QdrantClient(":memory:")` with `QdrantClient(path="/content/qdrant_storage")`.


In [ ]:
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
COLLECTION_NAME = "repo_code"
VECTOR_DIM = 384

print(f"Loading embedding model: {EMBEDDING_MODEL_NAME} ...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print("Embedding model ready.")

qdrant_client = QdrantClient(":memory:")
qdrant_client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=VECTOR_DIM, distance=Distance.COSINE),
)
print(f"Qdrant in-memory collection '{COLLECTION_NAME}' created.")

Loading embedding model: all-MiniLM-L6-v2 ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model ready.
Qdrant in-memory collection 'repo_code' created.


/tmp/ipykernel_785/2235124209.py:10: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant_client.recreate_collection(


## Step 1 — Clone the Repository

This cell clones the GitHub repository into `/content/repo`.

Progress is printed so you can see what is happening. For large repositories, this can take a moment.

If you are using a local folder, this step is skipped and the local path is used directly.


In [ ]:
CLONE_DIR = Path("/content/repo")


def clone_or_use_local() -> Path:
    """Clone the GitHub repo or validate the local folder path."""
    global CLONE_DIR

    if REPO_URL:
        if CLONE_DIR.exists():
            print(f"Removing existing clone at {CLONE_DIR} ...")
            shutil.rmtree(CLONE_DIR)

        print(f"Cloning {REPO_URL} ...")
        result = subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(CLONE_DIR)],
            capture_output=True,
            text=True,
        )
        if result.returncode != 0:
            raise RuntimeError(f"git clone failed:\n{result.stderr}")
        print(f"Clone complete → {CLONE_DIR}")
        return CLONE_DIR

    elif LOCAL_REPO_PATH:
        path = Path(LOCAL_REPO_PATH)
        if not path.exists():
            raise FileNotFoundError(f"Local path not found: {LOCAL_REPO_PATH}")
        print(f"Using local folder: {path}")
        return path

    else:
        raise ValueError("Set REPO_URL or LOCAL_REPO_PATH in the configuration cell.")


repo_path = clone_or_use_local()
print(f"\nRepository ready at: {repo_path}")

Cloning https://github.com/tiangolo/fastapi ...
Clone complete → /content/repo

Repository ready at: /content/repo


## Step 2 — Walk Files and Create Chunks

This cell walks every file in the cloned repository and splits each file into overlapping text chunks.

Supported file types: `.py`, `.js`, `.ts`, `.java`, `.go`, `.md`, `.txt`, and more.

Skipped directories: `node_modules`, `.git`, `build`, `dist`, `__pycache__`, and test folders.

Each chunk keeps metadata:
- `file_path` — which file it came from
- `component` — the top-level folder (e.g., `src`, `lib`)
- `chunk_index` — position within the file

Overlapping chunks (each chunk shares some lines with the previous one) helps avoid cutting important context at chunk boundaries.


In [ ]:
def should_skip(path: Path) -> bool:
    """Return True if this path is inside a directory we want to ignore."""
    for part in path.parts:
        if part in SKIP_DIRS or part.endswith(".egg-info"):
            return True
    return False


def chunk_text(text: str, size: int, overlap: int) -> List[str]:
    """Split text into overlapping character-level chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + size
        chunks.append(text[start:end])
        start += size - overlap
    return chunks


def walk_and_chunk(root: Path) -> List[dict]:
    """
    Walk the repository, read every supported file,
    and return a flat list of chunk dicts.
    """
    all_chunks = []
    file_count = 0
    skip_count = 0

    for filepath in sorted(root.rglob("*")):
        if not filepath.is_file():
            continue
        if should_skip(filepath.relative_to(root)):
            skip_count += 1
            continue
        if filepath.suffix.lower() not in SUPPORTED_EXTENSIONS:
            continue

        try:
            text = filepath.read_text(encoding="utf-8", errors="ignore").strip()
        except Exception:
            continue

        if not text or len(text) < 30:
            continue

        rel_path = str(filepath.relative_to(root))
        component = (
            filepath.relative_to(root).parts[0]
            if len(filepath.relative_to(root).parts) > 1
            else "root"
        )

        for idx, chunk in enumerate(chunk_text(text, CHUNK_SIZE, CHUNK_OVERLAP)):
            chunk = chunk.strip()
            if len(chunk) < 30:
                continue
            all_chunks.append(
                {
                    "text": chunk,
                    "file": rel_path,
                    "component": component,
                    "chunk_index": idx,
                }
            )

        file_count += 1

    print(f"Files indexed : {file_count}")
    print(f"Files skipped : {skip_count}")
    print(f"Total chunks  : {len(all_chunks)}")
    return all_chunks


print("Walking repository and chunking files...")
all_chunks = walk_and_chunk(repo_path)

Walking repository and chunking files...
Files indexed : 2548
Files skipped : 27
Total chunks  : 42581


## Step 3 — Embed Chunks and Index into Qdrant

This cell converts every chunk into an embedding vector and stores it in Qdrant.

Chunks are processed in batches to avoid running out of memory on large repositories.

Progress is printed every 200 chunks so you can track indexing on big repos.

After this cell runs, the knowledge base is ready. The agent can now retrieve relevant chunks for any query.


In [ ]:
def index_all_chunks(chunks: List[dict], batch_size: int = 64) -> None:
    """Embed chunks in batches and upsert into Qdrant."""
    total = len(chunks)
    upserted = 0

    for start in range(0, total, batch_size):
        batch = chunks[start : start + batch_size]
        texts = [c["text"] for c in batch]

        vectors = embedding_model.encode(
            texts, show_progress_bar=False, batch_size=batch_size
        ).tolist()

        points = [
            PointStruct(
                id=start + i,
                vector=vectors[i],
                payload={
                    "text": batch[i]["text"],
                    "file": batch[i]["file"],
                    "component": batch[i]["component"],
                    "chunk_index": batch[i]["chunk_index"],
                },
            )
            for i in range(len(batch))
        ]

        qdrant_client.upsert(collection_name=COLLECTION_NAME, points=points)
        upserted += len(points)

        if upserted % 200 == 0 or upserted == total:
            print(f"  Indexed {upserted}/{total} chunks...")

    print(f"Done. {upserted} chunks in Qdrant.")


print("Embedding and indexing — this may take 1-3 minutes for a medium repo...")
t0 = time.time()
index_all_chunks(all_chunks)
print(f"Total time: {time.time()-t0:.1f}s")

Embedding and indexing — this may take 1-3 minutes for a medium repo...
  Indexed 1600/42581 chunks...
  Indexed 3200/42581 chunks...
  Indexed 4800/42581 chunks...
  Indexed 6400/42581 chunks...
  Indexed 8000/42581 chunks...
  Indexed 9600/42581 chunks...
  Indexed 11200/42581 chunks...
  Indexed 12800/42581 chunks...
  Indexed 14400/42581 chunks...
  Indexed 16000/42581 chunks...
  Indexed 17600/42581 chunks...
  Indexed 19200/42581 chunks...


/tmp/ipykernel_785/3482121023.py:30: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20032 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  qdrant_client.upsert(collection_name=COLLECTION_NAME, points=points)


  Indexed 20800/42581 chunks...
  Indexed 22400/42581 chunks...
  Indexed 24000/42581 chunks...
  Indexed 25600/42581 chunks...
  Indexed 27200/42581 chunks...
  Indexed 28800/42581 chunks...
  Indexed 30400/42581 chunks...
  Indexed 32000/42581 chunks...
  Indexed 33600/42581 chunks...
  Indexed 35200/42581 chunks...
  Indexed 36800/42581 chunks...
  Indexed 38400/42581 chunks...
  Indexed 40000/42581 chunks...
  Indexed 41600/42581 chunks...
  Indexed 42581/42581 chunks...
Done. 42581 chunks in Qdrant.
Total time: 107.4s


## Retrieval Tool

This is the function the agent calls during each plan step.

It takes a query string, embeds it, and searches Qdrant for the most similar chunks.

Each result includes:
- File path — where the chunk came from
- Component — which top-level folder
- Similarity score — how relevant the chunk is
- Chunk text — the actual content

The agent calls this function once for each step in its plan and collects all results as observations.


In [ ]:
def retrieve(query: str, top_k: int = TOP_K) -> List[dict]:
    """Embed query and return top-k most similar chunks from Qdrant."""
    vector = embedding_model.encode([query], show_progress_bar=False)[0].tolist()
    results = qdrant_client.search(
        collection_name=COLLECTION_NAME, query_vector=vector, limit=top_k
    )
    return [
        {
            "text": r.payload["text"],
            "file": r.payload["file"],
            "component": r.payload["component"],
            "score": round(r.score, 4),
        }
        for r in results
    ]


# Smoke test
print("Retrieval smoke test...")
test_hits = retrieve("how does authentication work")
for h in test_hits:
    print(f"  [{h['component']}] {h['file']}  score={h['score']}")
    print(f"  {h['text'][:120]}...")
    print()

Retrieval smoke test...


AttributeError: 'QdrantClient' object has no attribute 'search'

## Agent Planner

This is where the agent decides what to do.

The planner takes the user's question and asks Gemini to produce a structured plan.

The plan lists specific retrieval queries — what to search for and why. Each query targets a different aspect of the original question.

Example:

**Question:** "How does authentication connect to user creation?"

**Generated plan:**
```
Step 1: Search for "authentication flow login token"
Step 2: Search for "user creation registration signup"
Step 3: Search for "shared validation service"
```

If the API call fails or the JSON is invalid, the planner falls back to a single-step plan using the original question as the query.


In [ ]:
def agent_planner(query: str) -> PlanExecutionResponse:
    """Generate a structured execution plan for the given query."""

    prompt = f"""You are a code repository analysis planner.

Given the question below, produce a JSON execution plan with EXACTLY this structure:

{{
  "plan": {{
    "objective": "<one-sentence goal>",
    "steps": [
      {{"step": 1, "action": "retrieve", "query": "<specific search string>"}},
      {{"step": 2, "action": "retrieve", "query": "<specific search string>"}},
      {{"step": 3, "action": "retrieve", "query": "<specific search string>"}}
    ],
    "target_components": ["<folder or module name>", "..."]
  }},
  "confidence_score": 0.90
}}

Rules:
- Generate 2 to 4 steps. Each query must be a specific, targeted search string.
- target_components lists relevant top-level folders or modules in the codebase.
- confidence_score is a float between 0.0 and 1.0.
- Return ONLY valid JSON. No markdown fences, no explanation.

Question: {query}
"""

    try:
        response = gemini_model.generate_content(
            prompt,
            generation_config=genai.GenerationConfig(
                response_mime_type="application/json"
            ),
        )
        data = json.loads(response.text.strip())
        return PlanExecutionResponse(**data)

    except Exception as e:
        print(f"[Planner fallback] {e}")
        return PlanExecutionResponse(
            plan=SearchPlan(
                objective=query,
                steps=[PlanStep(step=1, action="retrieve", query=query)],
                target_components=[],
            ),
            confidence_score=0.5,
        )


# Test
print("Planner test:")
tp = agent_planner("How is routing handled in this project?")
print(f"  Objective  : {tp.plan.objective}")
print(f"  Confidence : {tp.confidence_score}")
for s in tp.plan.steps:
    print(f"    Step {s.step}: {s.query}")

## Execution Engine — Agent Loop

This cell runs the plan step by step.

For each step:
1. Take the query from the plan
2. Call the retrieval tool
3. Collect the results as an observation

An `Observation` stores:
- Which step number produced it
- The query that was used
- A summary of the retrieved chunks
- How many chunks were retrieved

All observations together form the agent's working memory — the evidence it has gathered before generating the final answer.


In [ ]:
def execute_agent_loop(
    plan_response: PlanExecutionResponse, top_k: int = TOP_K, max_seconds: int = 90
) -> List[Observation]:
    """Run each plan step against the vector store and collect observations."""

    observations: List[Observation] = []
    start = time.time()

    for step in plan_response.plan.steps:
        if time.time() - start > max_seconds:
            print(f"[Agent Loop] Timeout after {max_seconds}s — stopping.")
            break

        print(f"  Step {step.step}: '{step.query}'")

        try:
            chunks = retrieve(step.query, top_k=top_k)

            if chunks:
                parts = [
                    f"[{c['component']}] {c['file']} (score={c['score']}):\n{c['text']}"
                    for c in chunks
                ]
                summary = "\n\n".join(parts)
            else:
                summary = f"No results found for: {step.query}"

            observations.append(
                Observation(
                    step=step.step,
                    query=step.query,
                    result_summary=summary,
                    chunks_found=len(chunks),
                )
            )

        except Exception as e:
            print(f"  [Step {step.step} error] {e}")
            observations.append(
                Observation(
                    step=step.step,
                    query=step.query,
                    result_summary=f"Retrieval error: {e}",
                    chunks_found=0,
                )
            )

    return observations

## Verification and Synthesis

**Verification**

Before calling Gemini, the agent checks that at least some evidence was collected.

If the observations are all empty (every retrieval step returned nothing), the agent stops and says so instead of sending an empty context to Gemini and getting a hallucinated answer.

**Synthesis**

All non-empty observations are packaged into a single context block. Gemini then generates the final answer from this evidence.

The answer is grounded — it comes from the retrieved chunks, not from Gemini's training data. The prompt tells Gemini to cite which files the answer comes from.


In [ ]:
def verify_observations(observations: List[Observation]) -> Tuple[bool, str]:
    """Check that observations contain usable evidence."""
    if not observations:
        return False, "No observations — retrieval returned nothing."
    total = sum(o.chunks_found for o in observations)
    if total == 0:
        return (
            False,
            "All steps returned 0 chunks. Try a different question or check the repo was indexed.",
        )
    return True, f"OK — {len(observations)} observations, {total} total chunks."


def synthesize_final_answer(
    query: str, observations: List[Observation], plan: SearchPlan
) -> str:
    """Ask Gemini to answer the question using retrieved observations."""

    context = "\n\n".join(
        f"=== Step {o.step} | Query: '{o.query}' ===\n{o.result_summary}"
        for o in observations
    )

    prompt = f"""You are a code repository analysis assistant.

Answer the question using ONLY the retrieved code observations below.
Be specific. Mention file names and function/class names where relevant.
If the observations lack enough detail, say so clearly — do not guess.

Objective: {plan.objective}
Target components: {', '.join(plan.target_components) or 'general'}

Question:
{query}

Retrieved Observations:
{context}

Answer:
"""

    return gemini_model.generate_content(prompt).text.strip()

## Full Agentic RAG Pipeline

`run_agentic_rag()` connects all the pieces:

```
User question
    ↓
Planner (Gemini) → structured plan with N steps
    ↓
Agent loop → runs each step → retrieves chunks
    ↓
Observations (named evidence from each step)
    ↓
Verification (check evidence is not empty)
    ↓
Synthesis (Gemini) → final grounded answer
    ↓
Return: answer + trace of every step
```

The trace shows exactly what the agent planned, what it searched for, what it found at each step, and how it reached the final answer.


In [ ]:
def run_agentic_rag(query: str) -> Tuple[str, str]:
    """
    Run the full Agentic RAG pipeline.

    Returns:
        (answer, trace) — answer is the final response;
        trace shows the complete agent reasoning.
    """
    lines = []

    def log(msg=""):
        print(msg)
        lines.append(msg)

    log("=" * 60)
    log(f"QUESTION: {query}")
    log("=" * 60)

    # Stage 1: Plan
    log("\n[Stage 1] Planning...")
    pr = agent_planner(query)
    log(f"  Objective  : {pr.plan.objective}")
    log(f"  Confidence : {pr.confidence_score}")
    log(f"  Components : {', '.join(pr.plan.target_components) or 'general'}")
    for s in pr.plan.steps:
        log(f"    Step {s.step} [{s.action}]: {s.query}")

    # Stage 2: Execute
    log("\n[Stage 2] Retrieving...")
    obs = execute_agent_loop(pr)
    for o in obs:
        log(f"  Step {o.step}: {o.chunks_found} chunk(s) — '{o.query}'")

    # Stage 3: Verify
    log("\n[Stage 3] Verifying...")
    ok, msg = verify_observations(obs)
    log(f"  {msg}")

    if not ok:
        return f"Could not generate answer.\nReason: {msg}", "\n".join(lines)

    # Stage 4: Synthesize
    log("\n[Stage 4] Synthesizing answer...")
    answer = synthesize_final_answer(query, obs, pr.plan)
    log("[Done]")
    log("=" * 60)

    return answer, "\n".join(lines)


# Full test
print("\nRunning full pipeline test...")
ans, trace = run_agentic_rag("What is the entry point of this project?")
print("\n--- ANSWER ---")
print(ans)

## Gradio Interface

Run this cell to launch the interactive interface.

The interface has two panels:
- **Final Answer** — the synthesized response with source citations
- **Agent Trace** — a step-by-step log showing the plan, each retrieval, and the observations

Example questions to try:
- "How is user authentication handled?"
- "What does the main entry point do?"
- "How are database models defined?"
- "What external APIs does this project call?"

For best results, ask complete questions with context rather than single words like "auth" (too vague).


In [ ]:
import gradio as gr


def ask_agent_ui(query: str):
    if not query or not query.strip():
        return "Please enter a question.", ""
    return run_agentic_rag(query.strip())


with gr.Blocks(title="Agentic RAG — Real Repo") as demo:

    gr.Markdown(f"""
# Agentic RAG — Real Repository
**Indexed:** `{REPO_URL or LOCAL_REPO_PATH}`  |  **Chunks:** {len(all_chunks)}

Ask any question about the codebase. The agent will plan, retrieve, verify, and synthesize a grounded answer.
""")

    question_box = gr.Textbox(
        label="Your Question",
        placeholder="e.g. How is routing handled? What does the main entry point do?",
        lines=3,
    )
    run_btn = gr.Button("Run Agent", variant="primary")

    with gr.Row():
        answer_box = gr.Textbox(
            label="Final Answer", lines=15, interactive=False, scale=2
        )
        trace_box = gr.Textbox(
            label="Agent Trace (Plan + Steps + Verification)",
            lines=15,
            interactive=False,
            scale=1,
        )

    run_btn.click(
        fn=ask_agent_ui, inputs=[question_box], outputs=[answer_box, trace_box]
    )

    gr.Examples(
        examples=[
            ["What is the entry point of this project?"],
            ["How is routing handled?"],
            ["What dependencies does this project use?"],
            ["How are errors handled?"],
            ["What does the main module do?"],
            ["How is authentication implemented?"],
        ],
        inputs=question_box,
    )

demo.launch(share=True)

## Tips for Getting Good Results

**Ask complete questions:**
- Good: "How is user authentication handled in this codebase?"
- Too vague: "auth"

**Change the repository:**
Just update `REPO_URL` in the configuration cell and re-run from the clone step onward.

**Make the index persistent:**
Replace `QdrantClient(":memory:")` with:
```python
qdrant_client = QdrantClient(path="/content/qdrant_storage")
```
This saves the index to disk so you do not have to re-embed on every session.

**For large repos (50+ files):**
Increase `CHUNK_SIZE` to 600-800 for faster indexing with less granular chunks.

**Swap the embedding model for better accuracy:**
```python
EMBEDDING_MODEL_NAME = "all-mpnet-base-v2"  # 768-dim, slower but stronger
VECTOR_DIM = 768
```
Remember to update `VECTOR_DIM` and recreate the Qdrant collection if you change the model.
